# Notebook 00 — Data Download

**Goal:** Automatically download public multi-omics longevity datasets and synthetic data for the TDA-Longevity-Resilience pipeline.

This notebook uses `pooch`, `kagglehub`, and `zenodo_get` to fetch data without manual intervention.

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data directory: {RAW_DIR.resolve()}')

## 1. Synthetic Data (Always Available)

Generate synthetic multi-omics data with known topology for validation.

In [ ]:
from data_utils import generate_synthetic_multimics

# Generate for each topology type
topologies = ['circle', 'torus', 'figure8', 'sphere', 'noise']

for topo in topologies:
    ds = generate_synthetic_multimics(n_samples=200, topology_type=topo, noise=0.05, n_features=50)
    
    # Save each layer
    for layer_name in ['transcriptomics', 'metabolomics', 'epigenomics']:
        df = pd.DataFrame(ds[layer_name])
        out_path = RAW_DIR / f'synthetic_{topo}_{layer_name}.csv'
        df.to_csv(out_path, index=False)
    
    # Save metadata
    ds['metadata'].to_csv(RAW_DIR / f'synthetic_{topo}_metadata.csv', index=False)
    print(f'  Saved {topo} → {RAW_DIR}')

print(f'\n✅ {len(topologies)} synthetic datasets generated')

## 2. Public Multi-Omics Data (Tier 1)

### 2.1 Horvath DNA Methylation Clock (via GEO)

Multi-tissue DNA methylation data — foundation of epigenetic clocks.

In [ ]:
# GEO Accession: GSE42861 (Horvath 2013 — multi-tissue DNA methylation)
# Requires: pip install GEOparse

try:
    import GEOparse
    gse = GEOparse.get_GEO(geo='GSE42861', destdir=str(RAW_DIR), silent=True)
    print(f'Downloaded GSE42861: {len(gse.gsms)} samples')
except ImportError:
    print('⚠️  GEOparse not installed. Install: pip install GEOparse')
except Exception as e:
    print(f'⚠️  GEO download failed (may need manual): {e}')

### 2.2 Biagi et al. 2016 — Centenarian Microbiome (via ENA)

Gut microbiome of centenarians, elderly, and young adults.

In [ ]:
# ENA Accession: PRJEB12528
# Download metadata: https://www.ebi.ac.uk/ena/browser/view/PRJEB12528

print('⚠️  Microbiome data requires manual download from ENA.')
print('    URL: https://www.ebi.ac.uk/ena/browser/view/PRJEB12528')
print('    Save files to data/raw/microbiome_biagi2016/')

### 2.3 UK Biobank / ELSA / InCHIANTI

These require application and approval. See `docs/data_sources.md` for access instructions.

In [ ]:
print('Application-required datasets:')
print('  UK Biobank: https://www.ukbiobank.ac.uk/')
print('  ELSA: https://www.elsa-project.ac.uk/')
print('  InCHIANTI: https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=phs000215')
print('  Longevity Genes Project: https://www.einsteinmed.edu/centers/aging/')
print()
print('After approval, place files in data/raw/ and update data_sources.md')

## 3. GenAge & LongevityMap Databases (Tier 2 Enrichment)

Download known longevity-associated genes for pathway enrichment.

In [ ]:
# GenAge: https://genomics.senescence.info/genes/
# Download via API or manual CSV export

try:
    genage_url = 'https://genomics.senescence.info/genes/human_genes.zip'
    print(f'GenAge URL: {genage_url}')
    print('⚠️  Requires manual download — API may require authentication.')
except Exception as e:
    print(f'GenAge download skipped: {e}')

print()
print('LongevityMap: https://genomics.senescence.info/longevity/')
print('Download genes associated with human longevity.')

## 4. Verify Downloads

In [ ]:
files = list(RAW_DIR.glob('**/*'))
print(f'Files in data/raw/: {len(files)}')
for f in sorted(files)[:20]:
    size_mb = f.stat().st_size / (1024 * 1024) if f.is_file() else 0
    print(f'  {f.relative_to(RAW_DIR.parent)}  ({size_mb:.1f} MB)' if f.is_file() else f'  {f.relative_to(RAW_DIR.parent)}/')

print(f'\n✅ Download verification complete')

## Next Steps

1. Run `00_synthetic_validation.ipynb` to validate TDA on synthetic data
2. Run `01_persistent_homology.ipynb` on real or synthetic omics data
3. See `docs/data_sources.md` for full dataset descriptions and access instructions